In [22]:
import pandas as pd
import sklearn


In [23]:
train = pd.read_csv("basic_train_set.csv")
test = pd.read_csv("basic_test_set.csv")


In [24]:
y_train = train['ClosePrice']
y_test = test['ClosePrice']

feature_cols = ['Latitude', 'Longitude', 'LivingArea', 'ParkingTotal', 'LotSizeAcres', 'YearBuilt', 'BathroomsTotalInteger', 'BedroomsTotal', 'Stories', 'LotSizeArea', 'MainLevelBedrooms', 'GarageSpaces', 'AssociationFee', 'LotSizeSquareFeet', 'Levels_MultiSplit', 'NumLevels']

X_train = train[feature_cols]
X_test = test[feature_cols]

# Models to Test
We will be trying out both RandomForest and DecisionTree models. Considering the amount of variables here, we can rule out Decision Trees as a viable set, but we will entertain it for completeness.

In [25]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score, mean_absolute_percentage_error


# 3. Train Decision Tree
# max_depth keeps the tree from growing indefinitely and overfitting
dt_model = DecisionTreeRegressor(max_depth=3, random_state=42)
dt_model.fit(X_train, y_train)

dt_preds = dt_model.predict(X_test)
print(f"Decision Tree R2: {r2_score(y_test, dt_preds):.3f}, MAE: {mean_absolute_error(y_test, dt_preds):,.0f}, MAPE: {mean_absolute_percentage_error(y_test, dt_preds):.2%}")

# 4. Train Random Forest
# n_jobs=-1 uses all available cores; max_depth bounds tree size so training doesn't blow up
# n_estimators is the number of individual trees in the forest
rf_model = RandomForestRegressor(n_estimators=100, max_depth=15, n_jobs=-1, random_state=42, verbose=1)
rf_model.fit(X_train, y_train)

rf_preds = rf_model.predict(X_test)
print(f"Random Forest R2: {r2_score(y_test, rf_preds):.3f}, MAE: {mean_absolute_error(y_test, rf_preds):,.0f}, MAPE: {mean_absolute_percentage_error(y_test, rf_preds):.2%}")

Decision Tree R2: 0.374, MAE: 331,854, MAPE: 36.20%


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 22 concurrent workers.
[Parallel(n_jobs=-1)]: Done   6 tasks      | elapsed:    1.9s


Random Forest R2: 0.892, MAE: 121,133, MAPE: 11.06%


[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:   10.3s finished
[Parallel(n_jobs=22)]: Using backend ThreadingBackend with 22 concurrent workers.
[Parallel(n_jobs=22)]: Done   6 tasks      | elapsed:    0.0s
[Parallel(n_jobs=22)]: Done 100 out of 100 | elapsed:    0.0s finished


In [26]:
import numpy as np
from sklearn.metrics import mean_absolute_percentage_error

# 5. Random Forest on log-transformed target
# ClosePrice is right-skewed, so training on log1p(price) and inverting with expm1
# tends to fit better and reduce error on high-end homes
y_train_log = np.log1p(y_train)

rf_log_model = RandomForestRegressor(n_estimators=100, max_depth=15, n_jobs=-1, random_state=42, verbose=1)
rf_log_model.fit(X_train, y_train_log)

rf_log_preds = np.expm1(rf_log_model.predict(X_test))
print(f"Random Forest (log target) R2: {r2_score(y_test, rf_log_preds):.3f}, MAE: {mean_absolute_error(y_test, rf_log_preds):,.0f}, MAPE: {mean_absolute_percentage_error(y_test, rf_log_preds):.2%}")

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 22 concurrent workers.
[Parallel(n_jobs=-1)]: Done   6 tasks      | elapsed:    1.9s


Random Forest (log target) R2: 0.877, MAE: 126,892, MAPE: 11.18%


[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:   10.8s finished
[Parallel(n_jobs=22)]: Using backend ThreadingBackend with 22 concurrent workers.
[Parallel(n_jobs=22)]: Done   6 tasks      | elapsed:    0.0s
[Parallel(n_jobs=22)]: Done 100 out of 100 | elapsed:    0.0s finished


# Notes 
-For baseline tests, it seems like randomforest performs significantly better than decision tree on all metrics except MBAPE
-
